## 14 多路召回融合难？动态阈值机制确保高质量结果优先排序

## 一、多路召回与加权融合：从单一到多维的检索优化

### 1.1 传统做法的局限性

在传统的检索流程中，通常遵循“单一查询 → 向量检索 → 排序 → 生成答案”的模式。这种方法虽然直接，但在面对复杂或多意图查询时，其召回能力显得捉襟见肘，极易遗漏高相关但直接匹配度不高的文档。

### 1.2 优化做法的显著优势

为了大幅提升检索效果，优化的多路召回策略引入了多维度的检索范式：多个扩展查询 → 多次检索 → 聚合结果 → 重排序 → 生成答案

这种方法的核心在于：它不再局限于原始查询本身，而是通过查询扩展（Query Expansion）生成多个相关查询，并并行地在不同召回路径中进行搜索（例如，基于关键词的检索、基于向量相似度的检索、基于知识图谱的检索等）。这样做带来的显著优势包括：

- **提高召回覆盖率（Recall）**：通过从多角度、多数据源进行检索，能捕获更多与用户意图相关的文档，从而大大减少漏召的可能性。
- **利用 LLM 进行更精细的 Reranking**：将多路召回到的候选文档集合起来，再利用强大的**大型语言模型（LLM）**对这些候选段落进行更精细的相关性打分和排序。LLM凭借其强大的语义理解能力，能识别文档与查询之间更深层次的关联，显著提升最终排序的准确性。

---

## 二、伪代码示例：多路召回与 LLM 重排序

下面是一个简化的伪代码示例，展示了多路召回与 LLM 重排序的基本流程：

In [4]:
retrieved_docs = []
for q in expanded_queries: # 遍历每个扩展查询
    docs = vector_db.search(q, top_k=5) # 从向量数据库中检索前 K 个文档
    retrieved_docs.extend(docs) # 将检索到的文档添加到集合中

# 使用 LLM 对所有召回的文档进行相关性评分并排序
reranked_docs = rerank_with_llm(retrieved_docs, original_query) 

NameError: name 'expanded_queries' is not defined

## 二、引入基于置信度的动态融合阈值机制

尽管多路召回结合 LLM 重排序显著提升了检索效果，但在实际应用中，如何有效融合来自不同召回路径的结果，并确保高质量结果的优先排序，仍是一个挑战。尤其当不同召回路径返回结果质量不一或存在重复文档时，简单的加权融合可能无法达到最优效果。

### 2.1 核心思想

为解决这一难题，我们可以引入一种**基于置信度的动态融合阈值机制**。这种机制的核心思想是：

1. **评估召回路径的置信度**：  
   为每个召回路径的结果赋予一个置信度分数。这个分数可以基于多种因素确定，例如：
   - **匹配度得分**：召回模型返回的相似度分数或匹配度。
   - **路径的历史稳定性与准确性**：通过离线评估或在线 A/B 测试，统计不同召回路径在历史数据中的表现。
   - **召回文档的质量特征**：例如文档的来源权威性、内容的完整性等。

2. **动态调整融合权重**：  
   根据召回路径的置信度分数，自动调整不同召回路径的权重。置信度高的路径，其返回的结果在融合时将获得更高的权重。这意味着系统会更倾向于采信那些被认为更可靠、更高质量的召回源。

3. **设置动态阈值**：  
   引入一个动态阈值，只有达到一定置信度或相关性分数门槛的文档才会被纳入最终的重排序列表。这个阈值可以根据整体召回结果分布、查询复杂程度，甚至是用户历史行为动态调整。例如，当召回的整体质量普遍较高时，可适当提高阈值以筛选更优质结果；反之，在召回结果较少或质量不佳时，可适当降低阈值以确保一定的召回数量。

---

### 2.2 引入机制的目标

通过引入基于置信度的动态融合阈值机制，我们能实现以下关键目标：

- ✅ **确保高质量结果优先排序**：  
  高置信度召回路径中的高质量文档将获得更高的融合权重，并在动态阈值的筛选下优先进入 LLM 重排序阶段。这意味着系统能够更有效地将真正相关的、有价值的信息推到用户面前。

- ✅ **提升排序稳定性**：  
  动态调整权重和阈值，使得系统能更好地适应不同查询类型和数据分布。即使某些召回路径在特定情况下表现不佳，整体融合机制也能通过降低其权重，避免其对最终排序结果产生负面影响，从而提升整个检索系统的稳定性。

---

### 2.3 演示代码：多路召回与 LLM 重排序

为了更好地理解上述概念，这里提供一个可运行的 Python 演示代码。该代码使用 [LlamaIndex](https://www.llamaindex.ai/ ) 框架与 OpenAI 大模型进行交互，展示了多路召回、文档去重以及重排序过程。


In [14]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL")

# 显式传入 api_key 和 api_base
Settings.llm = OpenAI(
    api_key=api_key, 
    api_base=base_url,
    model="gpt-4o"
)
Settings.embed_model = OpenAIEmbedding(
    api_key=api_key, 
    api_base=base_url,
    model_name="text-embedding-3-small"
)

In [10]:
from llama_index.core import VectorStoreIndex, SimpleKeywordTableIndex, Document
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.schema import NodeWithScore
from typing import List
from openai import OpenAI
import os

client = OpenAI(
    api_key=api_key,
    base_url=base_url
)

documents = [
    # AI / ML 基础
    Document(text="人工智能（AI）是一种旨在模拟、延伸和扩展人类智能的理论、方法、技术及应用系统的一门新的技术科学。", metadata={"source": "wiki"}),
    Document(text="机器学习是人工智能的一个子领域，通过让计算机从数据中自动学习规律，而无需显式编程。", metadata={"source": "wiki"}),
    Document(text="深度学习是机器学习研究中的一个新的领域，其动机在于建立、模拟人脑进行分析学习的神经网络。", metadata={"source": "wiki"}),
    Document(text="卷积神经网络（CNN）是深度学习中的一种重要架构，广泛应用于图像识别、目标检测等计算机视觉任务。", metadata={"source": "wiki"}),
    Document(text="循环神经网络（RNN）擅长处理序列数据，常用于自然语言处理、语音识别和时间序列预测。", metadata={"source": "wiki"}),
    Document(text="Transformer 架构由注意力机制（Attention Mechanism）构成，彻底改变了自然语言处理领域，是现代大模型的基础。", metadata={"source": "tech_blog"}),
    # 大型语言模型
    Document(text="LLM 是大型语言模型的缩写，它是一种经过大量文本数据训练的人工智能模型，广泛用于自然语言处理任务。", metadata={"source": "tech_blog"}),
    Document(text="GPT（生成式预训练变换器）是 OpenAI 开发的大型语言模型系列，GPT-4 在推理、编码和创意写作方面表现卓越。", metadata={"source": "tech_blog"}),
    Document(text="BERT 是 Google 提出的双向 Transformer 语言模型，通过掩码语言建模进行预训练，在问答和文本分类任务上取得了突破性进展。", metadata={"source": "tech_blog"}),
    Document(text="提示工程（Prompt Engineering）是设计和优化输入提示以引导大型语言模型产生期望输出的技术，是与 LLM 交互的核心技能。", metadata={"source": "tech_blog"}),
    # RAG 相关
    Document(text="检索增强生成（RAG）将信息检索与语言模型生成相结合，让模型在回答时能够参考外部知识库，有效减少幻觉问题。", metadata={"source": "paper"}),
    Document(text="向量数据库（Vector Database）用于存储和检索高维向量，是 RAG 系统中语义检索的核心组件，常见的有 Faiss、Pinecone、Chroma 等。", metadata={"source": "tech_blog"}),
    Document(text="文本嵌入（Text Embedding）将文字转换为稠密向量表示，使得语义相似的文本在向量空间中距离相近，是语义搜索的基础。", metadata={"source": "paper"}),
    Document(text="重排序（Reranking）是在初步召回后，使用更精确的模型对候选文档重新评分排序，以提高最终结果的准确性。", metadata={"source": "paper"}),
    Document(text="混合检索结合了稀疏检索（如 BM25）和密集向量检索的优势，能同时捕获关键词匹配和语义相似性，提升召回效果。", metadata={"source": "paper"}),
    # 自然语言处理
    Document(text="命名实体识别（NER）是 NLP 中的基础任务，用于从文本中识别并分类人名、地名、组织机构名等实体。", metadata={"source": "wiki"}),
    Document(text="情感分析是对文本中的主观信息进行提取和分类的任务，常用于分析用户评论、社交媒体内容中的情绪倾向。", metadata={"source": "wiki"}),
    Document(text="词嵌入（Word Embedding）技术如 Word2Vec 和 GloVe，通过将词语映射到低维连续向量空间，捕获词语之间的语义关系。", metadata={"source": "wiki"}),
]

# 基于文档，分别构建两种不同类型的索引：向量索引和关键词索引
# 向量索引擅长理解语义相关性（意思相近）
vector_index = VectorStoreIndex.from_documents(documents)
# 关键词索引擅长精确匹配文本中的词语
keyword_index = SimpleKeywordTableIndex.from_documents(documents)


class MultiPathRetriever(BaseRetriever):
    """
    自定义一个融合了向量搜索和关键词搜索的混合检索器。
    这种方法结合了两种搜索方式的优点，可以提高召回文档的全面性和准确性。
    """
    def __init__(self, vector_retriever, keyword_retriever):
        self.vector_retriever = vector_retriever
        self.keyword_retriever = keyword_retriever
        super().__init__()

    def _retrieve(self, query: str) -> List[NodeWithScore]:
        vector_nodes = self.vector_retriever.retrieve(query)
        keyword_nodes = self.keyword_retriever.retrieve(query)

        for node in vector_nodes:
            node.score = (node.score or 1.0) * 0.8
        for node in keyword_nodes:
            node.score = (node.score or 1.0) * 0.6

        combined_nodes = vector_nodes + keyword_nodes

        seen_texts = set()
        unique_nodes = []
        for node in combined_nodes:
            if node.text not in seen_texts:
                seen_texts.add(node.text)
                unique_nodes.append(node)
        
        scores = [n.score for n in unique_nodes if n.score is not None]
        if not scores:
            return []
        
        mean_score = sum(scores) / len(scores)
        std_dev = (sum((s - mean_score) ** 2 for s in scores) / len(scores)) ** 0.5
        threshold = mean_score * 0.8 + std_dev

        print(f"召回 {len(unique_nodes)} 个候选文档，均值={mean_score:.3f}，标准差={std_dev:.3f}，动态阈值={threshold:.3f}")
        filtered_nodes = [n for n in unique_nodes if n.score is not None and n.score >= threshold]
        print(f"动态阈值过滤后保留 {len(filtered_nodes)} 个文档")

        return filtered_nodes
    
retriever = MultiPathRetriever(
    vector_retriever=vector_index.as_retriever(similarity_top_k=5),
    keyword_retriever=keyword_index.as_retriever(similarity_top_k=5)
)

query = "什么是大型语言模型"

nodes = retriever.retrieve(query)

# LLM 重排序 (Reranking)
# 这是提升最终结果质量的关键一步。
# 我们让一个强大的语言模型（如GPT）来评估每个召回的文档与原始问题的相关性，并给出一个分数。
# 然后根据这个分数对文档重新排序。
reranked_nodes = []
import time
if nodes:
    node_scores = []
    print(f"\n开始对 {len(nodes)} 个节点进行重排序打分...")
    
    for idx, x in enumerate(nodes):
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": "你是一个相关性评估专家。仅输出一个 0-10 之间的数字，不要输出其他任何字符。"},
                    {"role": "user", "content": f"请评估以下段落与问题 '{query}' 的相关性，仅需给出一个0到10之间的分数。\n\n段落：{x.text}"}
                ],
                max_tokens=5,
                temperature=0
            )
            score_str = response.choices[0].message.content.strip()
            score = float(score_str) if score_str else 0.0
            print(f"节点 {idx+1} 得分: {score} | {x.text[:30]}...")
            
        except Exception as e:
            print(f"节点 {idx+1} 打分失败: {e}，默认设为 0 分")
            score = 0.0
            
        node_scores.append((score, x))
        time.sleep(1) 
        
    node_scores.sort(key=lambda item: item[0], reverse=True)
    reranked_nodes = [item[1] for item in node_scores]

print("\n--- 最终召回并重排序的文档 ---")
if not reranked_nodes:
    print("未能找到相关的文档。")
else:
    for i, node in enumerate(reranked_nodes):
        print(f"{i+1}. [召回分数: {node.score:.3f}] {node.text}")


召回 5 个候选文档，均值=0.396，标准差=0.056，动态阈值=0.372
动态阈值过滤后保留 2 个文档

开始对 2 个节点进行重排序打分...
节点 1 得分: 10.0 | LLM 是大型语言模型的缩写，它是一种经过大量文本数据训练的...
节点 2 得分: 8.0 | Transformer 架构由注意力机制（Attention...

--- 最终召回并重排序的文档 ---
1. [召回分数: 0.505] LLM 是大型语言模型的缩写，它是一种经过大量文本数据训练的人工智能模型，广泛用于自然语言处理任务。
2. [召回分数: 0.386] Transformer 架构由注意力机制（Attention Mechanism）构成，彻底改变了自然语言处理领域，是现代大模型的基础。


In [16]:
import os
from typing import List, Dict, Any
from llama_index.core.retrievers import BaseRetriever
from llama_index.core import Document, QueryBundle
from llama_index.core.schema import NodeWithScore
from llama_index.llms.openai import OpenAI as LLamaOpenAI


class MockRetriever(BaseRetriever):
    """
    一个模拟的检索器，用于演示目的。
    在真实场景中，这里应该是来自不同来源的多个真实检索器，
    例如：向量数据库检索器、关键词检索器、图数据库检索器等。
    """
    def __init__(self, all_docs: List[Document]):
        self._all_docs = all_docs
        super().__init__()

    def _retrieve(self, query_buddle: QueryBundle) -> List[NodeWithScore]:
        """
        模拟检索过程。
        为了简化，这里仅通过简单的关键词匹配来返回文档。
        """
        query_text = query_buddle.query_str.lower()
        retrieved_nodes = []
        for doc in self._all_docs:
            if any(keyword in doc.text.lower() for keyword in query_text.split()):
                retrieved_nodes.append(NodeWithScore(node=doc, score=1.0))
        
        return retrieved_nodes[:5]

def retrieve_and_rerank(
        original_query: str,
        expanded_queries: List[str],
        all_docs: List[Document]
) -> List[Dict[str, Any]]:
    """
    执行多路召回，聚合结果，然后使用 LLM 进行重排序。

    Args:
        original_query (str): 用户的原始查询。
        expanded_queries (List[str]): 由原始查询扩展出的一组子查询。
        all_docs (List[Document]): 知识库中的所有文档。

    Returns:
        List[Dict[str, Any]]: 经过重排序并带有分数的文档列表。
    """
    print(f"原始查询: '{original_query}'")
    print(f"扩展查询: {expanded_queries}\n")

    retriever = MockRetriever(all_docs)

    retrieved_docs = {}
    for q in expanded_queries:
        print(f"--- 正在执行召回, 查询: '{q}' ---")

        nodes_from_path = retriever.retrieve(QueryBundle(query_str=q))
        print(f" -> 召回了 {len(nodes_from_path)} 篇文档。")
        for node_with_score in nodes_from_path:
            doc = node_with_score.node
            retrieved_docs[doc.id_] = doc
    
    unique_docs = list(retrieved_docs.values())
    print(f"\n多路召回共聚合了 {len(unique_docs)} 篇不重复的文档。\n")

    if not unique_docs:
        return []
    
    print("--- 开始使用 LLM 对召回的文档进行重排序 ---")
    llm = LLamaOpenAI(model="gpt-4o", temperature=0.0, api_key=api_key, api_base=base_url)

    reranked_results = []
    for doc in unique_docs:
        prompt = (
            f"你是一个严格的相关性评估专家。请评估以下文档内容与用户查询的相关性。\n"
            f"【严格要求】：只输出一个 0 到 100 之间的整数（100表示最相关），绝对不要输出任何标点符号或其他解释性文字！！！\n\n"
            f"用户查询: '{original_query}'\n"
            f"文档内容: '{doc.text}'"
        )

        try:
            response = llm.complete(prompt)

            import re
            match = re.search(r'\d+', response.text)
            score = int(match.group()) if match else 0

            print(f"  - 正在评分: 文档ID '{doc.id_}' ... 分数: {score}")
            reranked_results.append({"doc": doc, "score": score})
        except Exception as e:
            print(f"  - 评分失败: 文档ID '{doc.id_}', 错误: {e}。记为0分。")
            reranked_results.append({"doc": doc, "score": 0})
        
        import time
        time.sleep(1)

    reranked_results.sort(key=lambda x: x["score"], reverse=True)

    return reranked_results

if __name__ == "__main__":
    # 模拟知识库中的文档
    mock_docs = [
        Document(id_="doc1", text="Python是一种广泛使用的编程语言，常用于数据科学和机器学习。"),
        Document(id_="doc2", text="机器学习是人工智能的一个分支，专注于让计算机从数据中学习。"),
        Document(id_="doc3", text="如何使用LlamaIndex进行文档索引和检索？LlamaIndex是一个用于将LLM与外部数据连接的数据框架。"),
        Document(id_="doc4", text="自然语言处理(NLP)是人工智能的另一个领域，涉及计算机理解和生成人类语言。"),
        Document(id_="doc5", text="Python的语法非常简洁和易读，适合初学者入门。"),
        Document(id_="doc6", text="大型语言模型(LLM)是近年来人工智能领域的重要突破，它们能够生成连贯且相关的文本。"),
        Document(id_="doc7", text="深度学习是机器学习的一个子集，使用神经网络进行数据建模。"),
        Document(id_="doc8", text="使用OpenAI API可以访问各种强大的LLM模型，进行文本生成、摘要和问答等任务。"),
    ]

    original_query = "关于LlamaIndex和大型语言模型的信息"
    expanded_queries = [
        "LlamaIndex 如何工作?",
        "大型语言模型应用",
        "LlamaIndex 和 LLM",
        "OpenAI API 使用",
    ]

    final_results = retrieve_and_rerank(original_query, expanded_queries, mock_docs)

    # 打印最终排序结果
    print("\n--- 最终重排序结果 (按LLM评分降序) ---")
    if not final_results:
        print("未能找到任何相关文档。")
    else:
        for i, item in enumerate(final_results):
            doc = item['doc']
            score = item['score']
            print(f"{i+1}. [分数: {score}] (ID: {doc.id_}) 内容: '{doc.text[:100]}...'")


原始查询: '关于LlamaIndex和大型语言模型的信息'
扩展查询: ['LlamaIndex 如何工作?', '大型语言模型应用', 'LlamaIndex 和 LLM', 'OpenAI API 使用']

--- 正在执行召回, 查询: 'LlamaIndex 如何工作?' ---
 -> 召回了 1 篇文档。
--- 正在执行召回, 查询: '大型语言模型应用' ---
 -> 召回了 0 篇文档。
--- 正在执行召回, 查询: 'LlamaIndex 和 LLM' ---
 -> 召回了 5 篇文档。
--- 正在执行召回, 查询: 'OpenAI API 使用' ---
 -> 召回了 4 篇文档。

多路召回共聚合了 7 篇不重复的文档。

--- 开始使用 LLM 对召回的文档进行重排序 ---
  - 正在评分: 文档ID 'doc3' ... 分数: 100
  - 正在评分: 文档ID 'doc1' ... 分数: 0
  - 正在评分: 文档ID 'doc4' ... 分数: 0
  - 正在评分: 文档ID 'doc5' ... 分数: 0
  - 正在评分: 文档ID 'doc6' ... 分数: 50
  - 正在评分: 文档ID 'doc7' ... 分数: 0
  - 正在评分: 文档ID 'doc8' ... 分数: 50

--- 最终重排序结果 (按LLM评分降序) ---
1. [分数: 100] (ID: doc3) 内容: '如何使用LlamaIndex进行文档索引和检索？LlamaIndex是一个用于将LLM与外部数据连接的数据框架。...'
2. [分数: 50] (ID: doc6) 内容: '大型语言模型(LLM)是近年来人工智能领域的重要突破，它们能够生成连贯且相关的文本。...'
3. [分数: 50] (ID: doc8) 内容: '使用OpenAI API可以访问各种强大的LLM模型，进行文本生成、摘要和问答等任务。...'
4. [分数: 0] (ID: doc1) 内容: 'Python是一种广泛使用的编程语言，常用于数据科学和机器学习。...'
5. [分数: 0] (ID: doc4) 内容: '自然语言处理(NLP)是人工智能的另一个领域，涉及计算机理解和生成人类语言。...'
6. [分

## 总结

多路召回结合 LLM 重排序是提升信息检索系统性能的强大策略。

而基于置信度的动态融合阈值机制则能够有效解决多路召回结果融合的难题，更能确保高质量结果的优先排序，并显著提升整个系统的稳定性和鲁棒性。